In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from dotenv import load_dotenv
from pathlib import Path
import sys

In [3]:
import torch
import torch.nn as nn 
from torch.nn import functional as f

In [4]:
from transformers import AutoTokenizer

In [5]:
try:
    project_root = next(p for p in Path().cwd().resolve().parents if (p /"env.env").exists())
except StopIteration:
    # Fallback to current working directory if not found
    project_root = Path.cwd()

sys.path.insert(0, str(project_root))

print(f"Project Root: {project_root}")
load_dotenv((project_root /'env.env').resolve())
data_dir = project_root / "datasets"

Project Root: /coc/scratch/vchopra/learning_env/Deep-Learning


In [6]:
from model_training.framework.data.pre_train.causal_modelling.tinyshakespeare.data import return_data
from model_training.framework.data.pre_train.causal_modelling.dataloader import DataLoaderLite, CausalLanguageModellingDataset

### Step-1:- Setup Model and Configs 

In [7]:
MODEL_NAME, TOKENIZER_NAME = 'Qwen/Qwen3-0.6B-Base', 'Qwen/Qwen3-0.6B-Base'
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_NAME)
tokenizer.__dict__.keys()

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.68k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

dict_keys(['add_prefix_space', '_vocab', '_merges', '_tokenizer', '_add_bos_token', '_add_eos_token', '_should_update_post_processor', 'init_inputs', 'init_kwargs', 'name_or_path', '_processor_class', '_pad_token_type_id', 'verbose', '_special_tokens_map', '_extra_special_tokens', 'model_max_length', 'padding_side', 'truncation_side', 'model_input_names', 'clean_up_tokenization_spaces', 'clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output', 'split_special_tokens', '_in_target_context_manager', 'chat_template', 'response_template', 'chat_control_tokens', 'deprecation_warnings', 'backend', 'files_loaded', 'vocab_file'])

In [8]:
data_config = CausalLanguageModellingDataset(
    total_batch_size=524288,
    micro_batch_size=8,
    seq_len=1024
)
assert data_config.total_batch_size % (data_config.micro_batch_size * data_config.seq_len) ==0 , "Make sure total batch size is divisible"

In [10]:
data_loader= DataLoaderLite(enc_data=return_data(root_dataset_dir= data_dir, tokenizer=tokenizer),
                            batch_size=data_config.micro_batch_size, 
                            seq_len=data_config.seq_len, 
                            tokenizer=tokenizer)

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (301829 > 131072). Running this sequence through the model will result in indexing errors


Loaded 301829 tokens
1 epoch = 36 tokens


### Step-2:- Loading the Model and Configuration

In [11]:
from model_training.framework.model.load_model.qwen3.config import Qwen3ModelConfig

In [12]:
QWEN_CONFIG_06_B = {
    "vocab_size": 151_936,     # Vocabulary size
    "context_len": 40_960,  # Length originally used during training
    "emb_dim": 1024,           # Embedding dimension
    "n_heads": 16,             # Number of attention heads
    "n_layers": 28,            # Number of layers
    "hidden_dim": 3072,        # Size of intermediate dim in FeedForward
    "head_dim": 128,           # Size of the heads in GQA
    "qk_norm": True,           # Whether to normalize queries & keys in GQA
    "n_kv_group": 8,          # Key-Value groups for GQA
    "rope_theta": 1_000_000.0,  # The base in RoPE's "theta"
    "dtype": torch.bfloat16,   # Lower-precision dtype to reduce memory
}

qwen_model_config = Qwen3ModelConfig(**QWEN_CONFIG_06_B)
qwen_model_config

Qwen3ModelConfig(vocab_size=151936, context_len=40960, emb_dim=1024, n_heads=16, n_layers=28, hidden_dim=3072, head_dim=128, qk_norm=True, n_kv_group=8, use_rope=True, rope_theta=1000000.0, dtype=torch.bfloat16, dropout=0.0)

In [13]:
from model_training.framework.model.load_model.qwen3.model import Qwen3Model

In [ ]:
qwen_model = Qwen3Model(qwen_model_config)
qwen_model.use_pretrained()

TypeError: Qwen3Model.use_pretrained() missing 1 required positional argument: 'hf_model'

: 

: 

: 

: 

: 